In [3]:
# GS - python
# Matheus Costa Cutrim - 568087 - CCPR - individual

# Sistema de Monitoramento da Missao Espacial Orion-X
# Projeto academico em Python com estruturas de dados, Pandas, Excel e API Web.

# Requisitos:
#    pip install pandas openpyxl requests

# Arquivos esperados:
#    base_missao_orion_x.xlsx

# Saida gerada:
#    relatorio_final_orion_x.xlsx


import pandas as pd
import requests
from datetime import datetime

ARQUIVO_ENTRADA = "base_missao_orion_x.xlsx"
ARQUIVO_SAIDA = "relatorio_final_orion_x.xlsx"


def exibir_estruturas_missao():
    """Demonstra uso de tuplas, dicionarios e lacos de repeticao."""
    print("\n================ ESTRUTURAS DE DADOS DA MISSAO ================")

    coordenadas_nave = (-23.5505, -46.6333, 408.0)  # latitude, longitude, altitude simulada em km

    sensores = {
        "SEN-001": {"tipo": "Temperatura", "modulo": "Habitat", "unidade": "Celsius"},
        "SEN-002": {"tipo": "Energia", "modulo": "Painel Solar", "unidade": "%"},
        "SEN-003": {"tipo": "Oxigenio", "modulo": "Suporte de Vida", "unidade": "%"},
        "SEN-004": {"tipo": "Radiacao", "modulo": "Escudo Externo", "unidade": "mSv"},
        "SEN-005": {"tipo": "Comunicacao", "modulo": "Antena Principal", "unidade": "%"},
    }

    modulos = {
        "MOD-HAB": "Modulo de habitacao",
        "MOD-ENE": "Modulo de energia",
        "MOD-SUP": "Modulo de suporte de vida",
        "MOD-COM": "Modulo de comunicacao",
    }

    print(f"Coordenadas atuais da nave: Latitude {coordenadas_nave[0]}, Longitude {coordenadas_nave[1]}, Altitude {coordenadas_nave[2]} km")

    print("\nSensores cadastrados:")
    for codigo, dados in sensores.items():
        print(f"- {codigo}: {dados['tipo']} | Modulo: {dados['modulo']} | Unidade: {dados['unidade']}")

    print("\nModulos da nave:")
    for codigo, descricao in modulos.items():
        print(f"- {codigo}: {descricao}")


def classificar_registro(linha):
    """Classifica cada registro da missao conforme limites simulados."""
    temperatura = linha["temperatura_c"]
    energia = linha["energia_pct"]
    oxigenio = linha["oxigenio_pct"]
    radiacao = linha["radiacao_msv"]
    comunicacao = linha["comunicacao_pct"]

    if temperatura >= 38 or energia < 30 or oxigenio < 70 or radiacao > 1.20 or comunicacao < 60:
        return "Critico"
    elif temperatura >= 32 or energia < 50 or oxigenio < 82 or radiacao > 0.80 or comunicacao < 75:
        return "Atencao"
    else:
        return "Normal"


def gerar_analise_automatica(linha):
    """Gera comentario automatico para cada linha monitorada."""
    problemas = []

    if linha["temperatura_c"] >= 32:
        problemas.append("temperatura elevada")
    if linha["energia_pct"] < 50:
        problemas.append("energia baixa")
    if linha["oxigenio_pct"] < 82:
        problemas.append("oxigenio reduzido")
    if linha["radiacao_msv"] > 0.80:
        problemas.append("radiacao acima do ideal")
    if linha["comunicacao_pct"] < 75:
        problemas.append("comunicacao instavel")

    if not problemas:
        return "Parametros dentro da faixa segura."

    return "Alerta: " + ", ".join(problemas) + "."


def buscar_dados_api():
    """Consome uma API publica, trata erros e retorna dados JSON tratados."""
    print("\n================ INTEGRACAO COM API WEB ================")

    url = "https://api.open-meteo.com/v1/forecast"
    parametros = {
        "latitude": 28.5721,
        "longitude": -80.6480,
        "current_weather": True,
    }

    try:
        resposta = requests.get(url, params=parametros, timeout=10)
        resposta.raise_for_status()
        dados_json = resposta.json()

        clima_atual = dados_json.get("current_weather", {})
        dados_tratados = {
            "fonte_api": "Open-Meteo",
            "local_referencia": "Centro Espacial Kennedy",
            "temperatura_externa_c": clima_atual.get("temperature"),
            "vento_kmh": clima_atual.get("windspeed"),
            "direcao_vento": clima_atual.get("winddirection"),
            "data_hora_api": clima_atual.get("time"),
        }

        print("Dados coletados da API:")
        for chave, valor in dados_tratados.items():
            print(f"- {chave}: {valor}")

        return dados_tratados

    except requests.exceptions.RequestException as erro:
        print("Nao foi possivel acessar a API no momento.")
        print(f"Erro identificado: {erro}")
        return {
            "fonte_api": "Open-Meteo",
            "local_referencia": "Centro Espacial Kennedy",
            "temperatura_externa_c": None,
            "vento_kmh": None,
            "direcao_vento": None,
            "data_hora_api": None,
            "observacao": "API indisponivel durante a execucao.",
        }


def processar_dados_missao():
    """Le arquivo Excel, manipula dados, analisa com Pandas e gera relatorio final."""
    print("\n================ LEITURA DO EXCEL ================")
    df = pd.read_excel(ARQUIVO_ENTRADA, sheet_name="Dados_Missao")
    print("Arquivo lido com sucesso!")

    print("\n================ HEAD ================")
    print(df.head())

    print("\n================ TAIL ================")
    print(df.tail())

    print("\n================ INFO ================")
    df.info()

    print("\n================ DESCRIBE ================")
    print(df.describe())

    print("\n================ MANIPULACAO E NOVAS COLUNAS ================")
    df["classificacao_risco"] = df.apply(classificar_registro, axis=1)
    df["analise_automatica"] = df.apply(gerar_analise_automatica, axis=1)
    df["energia_consumida_pct"] = 100 - df["energia_pct"]
    df["indice_estabilidade"] = (
        (df["energia_pct"] * 0.30)
        + (df["oxigenio_pct"] * 0.30)
        + (df["comunicacao_pct"] * 0.25)
        - (df["radiacao_msv"] * 10)
        - (df["temperatura_c"] * 0.15)
    ).round(2)

    registros_criticos = df[df["classificacao_risco"] == "Critico"]
    registros_atencao = df[df["classificacao_risco"] == "Atencao"]

    print("\nRegistros em estado critico:")
    print(registros_criticos)

    print("\nRegistros em atencao:")
    print(registros_atencao)

    resumo_estatistico = df.describe()
    resumo_classificacao = df["classificacao_risco"].value_counts().reset_index()
    resumo_classificacao.columns = ["classificacao_risco", "quantidade"]

    dados_api = buscar_dados_api()
    df_api = pd.DataFrame([dados_api])

    resumo_geral = pd.DataFrame({
        "indicador": [
            "Total de registros",
            "Registros normais",
            "Registros em atencao",
            "Registros criticos",
            "Temperatura media interna",
            "Energia media",
            "Oxigenio medio",
            "Radiacao media",
            "Comunicacao media",
            "Data de geracao do relatorio",
        ],
        "valor": [
            len(df),
            int((df["classificacao_risco"] == "Normal").sum()),
            int((df["classificacao_risco"] == "Atencao").sum()),
            int((df["classificacao_risco"] == "Critico").sum()),
            round(df["temperatura_c"].mean(), 2),
            round(df["energia_pct"].mean(), 2),
            round(df["oxigenio_pct"].mean(), 2),
            round(df["radiacao_msv"].mean(), 2),
            round(df["comunicacao_pct"].mean(), 2),
            datetime.now().strftime("%d/%m/%Y %H:%M:%S"),
        ]
    })

    print("\n================ RESUMO GERAL ================")
    print(resumo_geral)

    print("\n================ EXPORTACAO DO RELATORIO ================")
    with pd.ExcelWriter(ARQUIVO_SAIDA, engine="openpyxl") as writer:
        df.to_excel(writer, sheet_name="Dados_Analisados", index=False)
        resumo_estatistico.to_excel(writer, sheet_name="Resumo_Estatistico")
        resumo_classificacao.to_excel(writer, sheet_name="Resumo_Classificacao", index=False)
        registros_criticos.to_excel(writer, sheet_name="Alertas_Criticos", index=False)
        registros_atencao.to_excel(writer, sheet_name="Alertas_Atencao", index=False)
        df_api.to_excel(writer, sheet_name="Dados_API", index=False)
        resumo_geral.to_excel(writer, sheet_name="Resumo_Geral", index=False)

    print(f"Relatorio final gerado com sucesso: {ARQUIVO_SAIDA}")


def main():
    print("============================================================")
    print(" SISTEMA DE MONITORAMENTO DA MISSAO ESPACIAL ORION-X")
    print("============================================================")

    exibir_estruturas_missao()
    processar_dados_missao()

    print("\nProcessamento concluido com sucesso!")


if __name__ == "__main__":
    main()

 SISTEMA DE MONITORAMENTO DA MISSAO ESPACIAL ORION-X

================ ESTRUTURAS DE DADOS DA MISSAO ================
Coordenadas atuais da nave: Latitude -23.5505, Longitude -46.6333, Altitude 408.0 km

Sensores cadastrados:
- SEN-001: Temperatura | Modulo: Habitat | Unidade: Celsius
- SEN-002: Energia | Modulo: Painel Solar | Unidade: %
- SEN-003: Oxigenio | Modulo: Suporte de Vida | Unidade: %
- SEN-004: Radiacao | Modulo: Escudo Externo | Unidade: mSv
- SEN-005: Comunicacao | Modulo: Antena Principal | Unidade: %

Modulos da nave:
- MOD-HAB: Modulo de habitacao
- MOD-ENE: Modulo de energia
- MOD-SUP: Modulo de suporte de vida
- MOD-COM: Modulo de comunicacao

================ LEITURA DO EXCEL ================
Arquivo lido com sucesso!

================ HEAD ================
   registro_id           data_hora            modulo sensor_id  temperatura_c  \
0            1 2026-06-01 08:00:00           Habitat   SEN-001           24.5   
1            2 2026-06-01 11:00:00      Painel So